# Introduction

Projet **CRISP-DM — Prédiction de la faillite d'entreprise** : modèles supervisés sur indicateurs financiers, avec pipeline **`StandardScaler` → `SMOTE` → classifieur** (`imblearn` + scikit-learn).

**Livrables :** métriques comparées, figures et modèle sauvegardés dans `output/`, interprétation métier pour un rapport académique.

**Données :** fichier `data.csv` à la racine du projet (variable cible `Bankrupt?`). Vous pouvez aussi utiliser `output/data_cleaned.csv` après `analyse_donnees.py`.

**Chaîne CRISP-DM :** phase EDA / nettoyage (`analyse_donnees.py`, figures `eda_*.png`) puis cette phase **modélisation** (`class_distribution.png`, `correlation_heatmap.png`, etc.).


# Compréhension des données

Chargement avec *pandas*, structure du tableau, types et statistiques descriptives.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from IPython.display import display

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_STATE = 42
TARGET_COL = "Bankrupt?"
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data.csv"
OUTPUT_DIR = PROJECT_ROOT / "output"
TEST_SIZE = 0.2
CV_FOLDS = 5

OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(RANDOM_STATE)


In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
if TARGET_COL not in df.columns:
    raise ValueError(f"Colonne cible « {TARGET_COL} » absente. Colonnes : {list(df.columns)}")

print("Dimensions :", df.shape)
display(df.head(10))


In [ ]:
df.info()


In [ ]:
display(df.describe().T)


## Analyse exploratoire (EDA)

Valeurs manquantes, distribution de la cible, matrice de corrélation (variables numériques les plus liées à la faillite).


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
miss_df = pd.DataFrame({"manquantes": missing, "%": missing_pct})
print("Colonnes avec valeurs manquantes :")
display(miss_df[miss_df["manquantes"] > 0])
if miss_df["manquantes"].sum() == 0:
    print("Aucune valeur manquante.")


In [ ]:
def plot_class_distribution(y: pd.Series, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    counts = y.value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values, color=["#4C72B0", "#DD8452"])
    ax.set_xlabel("Bankrupt? (0 = non, 1 = oui)")
    ax.set_ylabel("Effectif")
    ax.set_title("Distribution de la variable cible (déséquilibre des classes)")
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width() / 2, v, str(v), ha="center", va="bottom")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


y_full = df[TARGET_COL].astype(int)
plot_class_distribution(y_full, OUTPUT_DIR / "class_distribution.png")
print("Proportion de faillites :", f"{y_full.mean() * 100:.2f}%")


In [ ]:
def plot_correlation_heatmap(X: pd.DataFrame, y: pd.Series, out_path: Path, top_k: int = 30) -> None:
    numeric = X.select_dtypes(include=[np.number])
    numeric = numeric.loc[:, numeric.nunique() > 1]
    if numeric.shape[1] == 0:
        return
    corr_target = numeric.corrwith(y).abs().sort_values(ascending=False)
    cols = corr_target.head(top_k).index.tolist()
    sub = numeric[cols]
    cm = sub.corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        square=True,
        linewidths=0.2,
        cbar_kws={"shrink": 0.6},
    )
    ax.set_title(
        f"Corrélation — {top_k} variables les plus corrélées (|ρ|) à la cible"
    )
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


X_all = df.drop(columns=[TARGET_COL])
plot_correlation_heatmap(X_all, y_full, OUTPUT_DIR / "correlation_heatmap.png")


# Préparation des données

Séparation **80 % / 20 %** **stratifiée** sur `Bankrupt?`. Aucune SMOTE sur le test : le rééchantillonnage est **uniquement dans le pipeline** d'entraînement (et dans chaque pli de validation croisée).


In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int)

if X.isnull().any().any():
    X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
n_minority = int(y_train.sum())
smote_kn = min(3, max(1, n_minority - 1))
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Faillites (train) :", n_minority, "| k_neighbors SMOTE :", smote_kn)


# Modélisation

Pipeline **`StandardScaler` → `SMOTE` → classifieur**. **GridSearchCV** (métrique **F1**) **uniquement pour la Random Forest**, comme exigé. Les autres modèles sont entraînés avec des hyperparamètres fixes raisonnables. **XGBoost** est ajouté si la bibliothèque est installée.


In [ ]:
def make_base_pipeline(classifier, smote_k_neighbors: int) -> ImbPipeline:
    return ImbPipeline(
        [
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k_neighbors)),
            ("clf", classifier),
        ]
    )


RF_PARAM_GRID = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 16, None],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__class_weight": [None, "balanced"],
}


def train_with_optional_grid(
    pipe: ImbPipeline,
    param_grid: Optional[Dict[str, List[Any]]],
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
) -> Tuple[Any, Optional[Dict[str, Any]]]:
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    if param_grid:
        gs = GridSearchCV(
            pipe,
            param_grid,
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        gs.fit(X_tr, y_tr)
        return gs.best_estimator_, gs.best_params_
    pipe.fit(X_tr, y_tr)
    return pipe, None


def evaluate_binary(model: Any, X_te: pd.DataFrame, y_te: pd.Series) -> Dict[str, float]:
    y_pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "precision": float(precision_score(y_te, y_pred, zero_division=0)),
        "recall": float(recall_score(y_te, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_te, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_te, proba)),
    }


configs: List[Dict[str, Any]] = [
    {
        "name": "Logistic Regression",
        "key": "logistic_regression",
        "pipe": make_base_pipeline(
            LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                solver="lbfgs",
            ),
            smote_kn,
        ),
        "grid": None,
    },
    {
        "name": "Decision Tree",
        "key": "decision_tree",
        "pipe": make_base_pipeline(
            DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                max_depth=12,
                min_samples_leaf=4,
                class_weight="balanced",
            ),
            smote_kn,
        ),
        "grid": None,
    },
    {
        "name": "Random Forest",
        "key": "random_forest",
        "pipe": make_base_pipeline(
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            smote_kn,
        ),
        "grid": RF_PARAM_GRID,
    },
    {
        "name": "Gradient Boosting",
        "key": "gradient_boosting",
        "pipe": make_base_pipeline(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                n_estimators=200,
                max_depth=3,
                learning_rate=0.1,
                min_samples_leaf=3,
            ),
            smote_kn,
        ),
        "grid": None,
    },
]

try:
    from xgboost import XGBClassifier

    configs.append(
        {
            "name": "XGBoost",
            "key": "xgboost",
            "pipe": make_base_pipeline(
                XGBClassifier(
                    random_state=RANDOM_STATE,
                    eval_metric="logloss",
                    n_estimators=200,
                    max_depth=4,
                    learning_rate=0.08,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    n_jobs=-1,
                ),
                smote_kn,
            ),
            "grid": None,
        }
    )
except ImportError:
    print("XGBoost non disponible — ignoré.")

rows: List[Dict[str, Any]] = []
fitted: Dict[str, Any] = {}
for cfg in configs:
    model, best_p = train_with_optional_grid(cfg["pipe"], cfg["grid"], X_train, y_train)
    metrics = evaluate_binary(model, X_test, y_test)
    rows.append(
        {
            "model": cfg["name"],
            "key": cfg["key"],
            **metrics,
            "best_params": json.dumps(best_p, ensure_ascii=False) if best_p else "",
        }
    )
    fitted[cfg["key"]] = model

results = pd.DataFrame(rows)
results.drop(columns=["key"], errors="ignore").to_csv(OUTPUT_DIR / "model_results.csv", index=False)
print("Sauvegardé :", OUTPUT_DIR / "model_results.csv")


# Évaluation et comparaison

Tri par **F1-score** puis **ROC-AUC**. Le meilleur estimateur est sauvegardé avec `joblib`.


In [ ]:
results_sorted = results.sort_values(
    ["f1_score", "roc_auc"], ascending=[False, False]
).reset_index(drop=True)
comparison = results_sorted[
    ["model", "accuracy", "precision", "recall", "f1_score", "roc_auc"]
]
display(comparison)

best_key = results_sorted.iloc[0]["key"]
best_name = results_sorted.iloc[0]["model"]
best_model = fitted[best_key]
joblib.dump(best_model, OUTPUT_DIR / "best_model.pkl")
print("Meilleur modèle :", best_name)
print("Fichier :", OUTPUT_DIR / "best_model.pkl")


# Résultats — visualisations (meilleur modèle)


In [ ]:
feature_names = feature_cols


def extract_feature_importance(model: Any, names: List[str]) -> np.ndarray:
    clf = model.named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        return np.asarray(clf.feature_importances_)
    if hasattr(clf, "coef_"):
        return np.asarray(np.abs(clf.coef_).ravel())
    return np.zeros(len(names))


def plot_confusion_matrix_fig(y_true, y_pred, out_path: Path, title: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Prédit 0", "Prédit 1"],
        yticklabels=["Réel 0", "Réel 1"],
    )
    ax.set_title(title)
    ax.set_ylabel("Vérité terrain")
    ax.set_xlabel("Prédiction")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


def plot_roc_fig(y_true, y_score, out_path: Path, title: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, label=f"ROC (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", label="Hasard")
    ax.set_xlabel("Taux de faux positifs")
    ax.set_ylabel("Taux de vrais positifs")
    ax.set_title(title)
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


def plot_feature_importance_fig(
    importances: np.ndarray, names: List[str], out_path: Path, top_n: int = 25
) -> None:
    order = np.argsort(importances)[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(10, 8))
    y_pos = np.arange(len(order))
    ax.barh(y_pos, importances[order], color="#4C72B0")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([names[i] for i in order], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Importance")
    ax.set_title(f"Top {top_n} variables — {best_name}")
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)


y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

plot_confusion_matrix_fig(
    y_test,
    y_pred_best,
    OUTPUT_DIR / "confusion_matrix.png",
    f"Matrice de confusion — {best_name}",
)
plot_roc_fig(
    y_test,
    y_proba_best,
    OUTPUT_DIR / "roc_curve.png",
    f"Courbe ROC — {best_name}",
)
imp = extract_feature_importance(best_model, feature_names)
plot_feature_importance_fig(imp, feature_names, OUTPUT_DIR / "feature_importance.png")


## Interprétation métier

### Déséquilibre des classes
Les **faillites** sont rares par rapport aux entreprises saines. Sans traitement, un modèle peut prédire presque toujours « non-faillite », obtenir une **accuracy élevée** et pourtant **rater la plupart des cas critiques**. La **SMOTE** sur le jeu d'entraînement (dans le pipeline) aide le modèle à mieux apprendre la classe minoritaire.

### Intérêt du F1-score
Le **F1** combine **précision** et **rappel** (moyenne harmonique). Il pénalise les modèles qui annoncent trop souvent à tort une faillite **ou** qui **ne détectent pas** les vraies faillites — ce qui est central quand les classes sont déséquilibrées.

### Impact des faux négatifs (faillite non détectée)
Un **faux négatif** : entreprise en difficulté classée comme saine. Conséquences possibles : **pertes de créance** (banques, fournisseurs), **mauvaise allocation du risque** dans un portefeuille, **surprise** pour les parties prenantes. Selon le contexte, maximiser le **rappel** peut être prioritaire.

### Pourquoi le modèle retenu est « le meilleur » ici
Le choix est **objectif** : classement sur le **jeu de test** selon le **F1**, puis la **ROC-AUC** en cas d'égalité. La **Random Forest** a en plus été **affinée par GridSearchCV** (validation croisée stratifiée, métrique F1). Ce meilleur modèle sert de candidat principal pour la suite (déploiement, analyse d'erreurs, seuil de décision métier).


# Conclusion

- Pipeline **StandardScaler → SMOTE → classifieur**, sans fuite du test dans le sur-échantillonnage.
- Comparaison de plusieurs algorithmes ; **réglage systématique par grille** sur la **Random Forest** (F1).
- Fichiers générés dans `output/` : `model_results.csv`, `best_model.pkl`, `class_distribution.png`, `correlation_heatmap.png`, `confusion_matrix.png`, `roc_curve.png`, `feature_importance.png`.

*Exécuter toutes les cellules dans l'ordre après avoir placé `data.csv` à la racine du projet.*


In [ ]:
best_metrics = {
    "f1_score": float(results_sorted.iloc[0]["f1_score"]),
    "roc_auc": float(results_sorted.iloc[0]["roc_auc"]),
    "recall": float(results_sorted.iloc[0]["recall"]),
    "precision": float(results_sorted.iloc[0]["precision"]),
}
md_text = f"""# Interprétation métier — export

## Synthèse quantitative (meilleur modèle : {best_name})

- F1 = **{best_metrics['f1_score']:.4f}** | ROC-AUC = **{best_metrics['roc_auc']:.4f}**
- Rappel = **{best_metrics['recall']:.4f}** | Précision = **{best_metrics['precision']:.4f}**

## Tableau comparatif

```
{comparison.to_string(index=False)}
```
"""
(OUTPUT_DIR / "business_interpretation.md").write_text(md_text, encoding="utf-8")
print("Export :", OUTPUT_DIR / "business_interpretation.md")
